# Individual-level EDA — Wearable Sensor Data

**Notebook 3 of 3** — per-patient feature engineering, fatigue-day identification, phase comparison, and cross-patient profiling.

| Section | Content |
|---------|--------|
| **0** | Data setup (identical to eda2_overall) |
| **1** | Per-day & per-patient feature engineering |
| **2** | Individual exploration tools |
| **3** | Fatigue day analysis |
| **4** | Phase analysis (first vs second half of study) |
| **5** | Cross-patient feature comparison |


---
## Section 0 — Data Setup


In [40]:
import glob, os, warnings
import plotly
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

DATA_DIR = '../data'
EXT_DIR  = '../data/external'
FIG_DIR  = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

DISEASE_ORDER   = ['Early Disease Stage', 'Fast Disease Progression', 'Late Disease Stage']
DISEASE_PALETTE = {
    'Early Disease Stage':      '#2ecc71',
    'Fast Disease Progression': '#e74c3c',
    'Late Disease Stage':       '#3498db',
}
def hex_rgba(hex_col, alpha=0.15):
    h = hex_col.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'
DISEASE_RGBA = {k: hex_rgba(v) for k, v in DISEASE_PALETTE.items()}
print(f'Setup complete — Plotly {plotly.__version__}, pandas {pd.__version__}')


Setup complete — Plotly 6.7.0, pandas 2.3.3


In [41]:
clinical = pd.read_csv(f'{DATA_DIR}/ClinicalMarkers_final.csv')
clinical.columns = clinical.columns.str.strip().str.lower().str.replace('.', '_', regex=False)
clinical['age'] = 2026 - clinical['age']
print(f'Clinical: {len(clinical)} participants')
print(clinical['disease_type'].value_counts().to_string())
clinical.head()


Clinical: 44 participants
disease_type
Late Disease Stage          19
Early Disease Stage         18
Fast Disease Progression     7


,record,age,sex,disease_type,id
0,1,55,Female,Late Disease Stage,3389
1,2,29,Female,Early Disease Stage,4349
2,3,45,Female,Late Disease Stage,6155
3,4,53,Female,Early Disease Stage,9173
4,5,45,Female,Early Disease Stage,2815


In [42]:
files = glob.glob(f'{DATA_DIR}/Hourly Sensor Data/RHourly_*.csv')
print(f'Found {len(files)} sensor files')
chunks = []
for fp in files:
    pid   = int(os.path.basename(fp).replace('RHourly_', '').replace('.csv', ''))
    chunk = pd.read_csv(fp)
    chunk['id'] = pid
    chunks.append(chunk)
sensor = pd.concat(chunks, ignore_index=True)
sensor['heartrate'] = pd.to_numeric(sensor['heartrate'], errors='coerce')
sensor['time']      = pd.to_datetime(sensor['time'])
sensor['date']      = sensor['time'].dt.floor('D')
sensor['hour']      = sensor['time'].dt.hour
sensor['week']      = sensor['time'].dt.isocalendar().week.astype(int)
sensor['month']     = sensor['time'].dt.month
sensor['weekday']   = sensor['time'].dt.dayofweek
print(f'Total rows: {len(sensor):,}  |  HR missing: {sensor["heartrate"].isna().sum():,}')
sensor.describe().round(1)


Found 44 sensor files
Total rows: 57,928  |  HR missing: 3,519


,time,steps,sleep,heartrate,id,date,hour,week,month,weekday
count,57928,57928.0,57928.0,54409.0,57928.0,57928,57928.0,57928.0,57928.0,57928.0
mean,2021-06-19 03:56:11.115868160,303.5,17.9,78.9,5721.4,2021-06-18 16:27:27.120563200,11.5,24.2,6.1,3.0
min,2021-01-08 00:00:00,0.0,0.0,45.5,1120.0,2021-01-08 00:00:00,0.0,1.0,1.0,0.0
25%,2021-04-17 11:00:00,0.0,0.0,68.9,3389.0,2021-04-17 00:00:00,5.0,15.0,4.0,1.0
50%,2021-06-10 19:00:00,83.9,0.0,77.7,6155.0,2021-06-10 00:00:00,11.0,23.0,6.0,3.0
75%,2021-09-02 21:00:00,368.4,60.0,87.6,7928.0,2021-09-02 00:00:00,17.0,35.0,9.0,5.0
max,2021-11-14 23:00:00,6125.0,60.0,139.4,9926.0,2021-11-14 00:00:00,23.0,45.0,11.0,6.0
std,NaN,560.6,26.5,13.4,2466.7,NaN,6.9,11.4,2.6,2.0


In [43]:
df = pd.merge(
    clinical[['id', 'age', 'sex', 'disease_type']],
    sensor, on='id', how='left'
)
print(f'Merged df: {df.shape}')
df.head(3)


Merged df: (57928, 13)


,id,age,sex,disease_type,time,steps,sleep,heartrate,date,hour,week,month,weekday
0,3389,55,Female,Late Disease Stage,2021-01-08 00:00:00,0.0,0,NaN,2021-01-08,0,1,1,4
1,3389,55,Female,Late Disease Stage,2021-01-08 01:00:00,0.0,0,NaN,2021-01-08,1,1,1,4
2,3389,55,Female,Late Disease Stage,2021-01-08 02:00:00,0.0,0,NaN,2021-01-08,2,1,1,4


In [44]:
# Shift the sleep column 6 hours forward within each patient so that
# summing sleep_lag6 over a calendar day captures sleep from 18:00 (prev day)
# sleep_lag6[t] = sleep[t - 6h] 
df = df.sort_values(['id', 'time']).reset_index(drop=True)
df['sleep_lag6'] = df.groupby('id')['sleep'].shift(6)

n_null = df['sleep_lag6'].isna().sum()
print(f'sleep_lag6: {df["sleep_lag6"].notna().sum():,} non-null, '
      f'{n_null:,} NaN (first 6h per patient have no prior sleep window)')


sleep_lag6: 57,664 non-null, 264 NaN (first 6h per patient have no prior sleep window)


In [47]:
# Complete-day filter: keep only patient-days with 24 hourly records,
# steps/sleep_lag6/heartrate all non-null (sleep_lag6 requires prior 6h of data).
_all_present = df[['steps', 'sleep_lag6', 'heartrate']].notna().all(axis=1)
_n_complete  = _all_present.groupby([df['id'], df['date']]).sum()
_n_total     = df.groupby(['id', 'date']).size()
complete_days = (
    pd.concat([_n_complete.rename('n_ok'), _n_total.rename('n_hrs')], axis=1)
    .query('n_ok == 24 and n_hrs == 24')
    .reset_index()[['id', 'date']]
)
_rows_before = len(df)
_days_before = df.groupby(['id', 'date']).ngroups
df = df.merge(complete_days, on=['id', 'date'], how='inner')
df['n_complete_days'] = df.groupby('id')['date'].transform('nunique')
print(f'Rows:         {_rows_before:,} -> {len(df):,}  (-{_rows_before - len(df):,})')
print(f'Patient-days: {_days_before:,} -> {df.groupby(["id","date"]).ngroups:,}')
print(f'Participants with >=1 complete day: {df["id"].nunique()} / 44')


Rows:         57,928 -> 45,984  (-11,944)
Patient-days: 2,422 -> 1,916
Participants with >=1 complete day: 42 / 44


In [48]:
WEATHER_LOCAL = f'{EXT_DIR}/weather/ogd-smn_klo_h_historical_2020-2029.csv'
try:
    weather_raw = pd.read_csv(WEATHER_LOCAL, sep=';', encoding='latin-1')
    print('Loaded weather from local file')
except FileNotFoundError:
    import urllib.request
    URL = 'https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/klo/ogd-smn_klo_h_historical_2020-2029.csv'
    urllib.request.urlretrieve(URL, WEATHER_LOCAL)
    weather_raw = pd.read_csv(WEATHER_LOCAL, sep=';', encoding='latin-1')
    print('Downloaded and saved')
weather_raw['datetime'] = pd.to_datetime(weather_raw['reference_timestamp'], format='%d.%m.%Y %H:%M')
w2021 = weather_raw[weather_raw['datetime'].dt.year == 2021].copy()
w2021 = w2021[['datetime', 'tre200h0', 'tre200hn', 'tre200hx',
                'rre150h0', 'sre000h0', 'ure200h0']].copy()
w2021.columns = ['datetime', 'temp_c', 'temp_c_min', 'temp_c_max',
                  'precip_mm', 'sunshine_min', 'humidity_pct']
w2021['date'] = w2021['datetime'].dt.floor('D')
weather_daily = w2021.groupby('date').agg(
    temp_max      =('temp_c_max', 'max'),
    temp_mean     =('temp_c',     'mean'),
    temp_min      =('temp_c_min', 'min'),
    precip_total  =('precip_mm',    'sum'),
    sunshine_total=('sunshine_min', 'sum'),
    humidity_mean =('humidity_pct', 'mean'),
).reset_index()
print(f'Weather daily: {len(weather_daily)} days')


Loaded weather from local file
Weather daily: 365 days


In [49]:
CALENDAR_LOCAL = f'{EXT_DIR}/holidays/zurich_calendar_2021.csv'
try:
    calendar = pd.read_csv(CALENDAR_LOCAL, parse_dates=['date'])
    print('Loaded calendar from local file')
except FileNotFoundError:
    import holidays as hol_lib
    zh_hols = hol_lib.country_holidays('CH', subdiv='ZH', years=2021)
    public_hols = pd.to_datetime(list(zh_hols.keys()))
    school_breaks = [
        ('2021-04-26','2021-05-08'), ('2021-07-19','2021-08-21'),
        ('2021-10-11','2021-10-23'), ('2021-12-20','2022-01-01'),
    ]
    calendar = pd.DataFrame({'date': pd.date_range('2021-01-01','2021-12-31',freq='D')})
    calendar['is_weekend']    = calendar['date'].dt.dayofweek >= 5
    calendar['is_public_hol'] = calendar['date'].isin(public_hols)
    calendar['is_school_break'] = False
    for s, e in school_breaks:
        calendar.loc[calendar['date'].between(s,e), 'is_school_break'] = True
    calendar['official_day_off'] = calendar['is_weekend'] | calendar['is_public_hol']
    prev_off = calendar['official_day_off'].shift(1, fill_value=False)
    next_off = calendar['official_day_off'].shift(-1, fill_value=False)
    calendar['is_bridge_day'] = ~calendar['official_day_off'] & ~calendar['is_weekend'] & prev_off & next_off
    def _day_type(row):
        if row['is_public_hol']:  return 'public_holiday'
        if row['is_bridge_day']:  return 'bridge_day'
        if row['is_weekend']:     return 'weekend'
        return 'workday'
    calendar['day_type'] = calendar.apply(_day_type, axis=1)
    calendar['is_extended_day_off'] = calendar['day_type'] != 'workday'
    calendar.to_csv(CALENDAR_LOCAL, index=False)
    print('Built and saved')
print(calendar['day_type'].value_counts().to_string())


Loaded calendar from local file
day_type
workday           255
weekend            97
public_holiday     12
bridge_day          1


In [50]:
COVID_LOCAL = f'{EXT_DIR}/covid/ch_stringency_2021.csv'
try:
    covid_ch = pd.read_csv(COVID_LOCAL, parse_dates=['date'])
    print('Loaded COVID data from local file')
except FileNotFoundError:
    import urllib.request
    URL = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'
    covid_all = pd.read_csv(URL)
    covid_ch  = covid_all[
        (covid_all['iso_code'] == 'CHE') &
        (covid_all['date'].between('2021-01-01', '2021-12-31'))
    ][['date', 'stringency_index', 'new_cases_smoothed_per_million']].copy()
    covid_ch['date'] = pd.to_datetime(covid_ch['date'])
    os.makedirs(f'{EXT_DIR}/covid', exist_ok=True)
    covid_ch.to_csv(COVID_LOCAL, index=False)
    print('Downloaded and saved')
print(f'Stringency range: {covid_ch["stringency_index"].min():.1f} - {covid_ch["stringency_index"].max():.1f}')


Loaded COVID data from local file
Stringency range: 44.4 - 60.2


In [54]:
daily = (
    df.groupby(['id', 'date', 'disease_type', 'sex', 'age'])
    .agg(
        n_complete_days = ('n_complete_days', 'first'),
        daily_steps     = ('steps',     'sum'),
        sleep_last_night = ('sleep_lag6', 'sum'),
        # daily_sleep      = ('sleep',       'sum'),  # raw midnight-to-midnight (kept for reference)
        mean_hr         = ('heartrate', 'mean'),
        active_hours    = ('steps',     lambda x: (x > 0).sum()),
    )
    .reset_index()
)
daily = daily.merge(weather_daily, on='date', how='left')
daily = daily.merge(
    calendar[['date','day_type','is_weekend','is_public_hol','is_school_break','is_extended_day_off']],
    on='date', how='left'
)
daily = daily.merge(covid_ch[['date','stringency_index']], on='date', how='left')
print(f'Daily: {len(daily):,} rows | {daily["id"].nunique()} participants')
print(f'Dates: {daily["date"].min().date()} -> {daily["date"].max().date()}')
daily.describe().round(1)


Daily: 1,916 rows | 42 participants
Dates: 2021-01-09 -> 2021-11-14


,id,date,age,n_complete_days,daily_steps,sleep_last_night,mean_hr,active_hours,temp_max,temp_mean,temp_min,precip_total,sunshine_total,humidity_mean,stringency_index
count,1916.0,1916,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0,1916.0
mean,5631.2,2021-06-19 08:40:50.104384256,48.6,47.7,7637.6,481.4,78.6,18.0,17.3,11.7,6.3,3.0,336.9,75.9,51.4
min,1120.0,2021-01-09 00:00:00,21.0,10.0,445.9,0.0,62.1,6.0,-5.2,-6.9,-14.8,0.0,0.0,43.5,44.4
25%,3389.0,2021-04-13 00:00:00,44.0,43.0,3814.2,431.8,73.1,17.0,12.5,6.8,1.1,0.0,98.8,67.6,48.2
50%,5977.0,2021-06-12 00:00:00,48.0,49.0,6800.9,490.0,77.5,18.0,17.8,11.5,6.9,0.0,290.5,77.4,50.0
75%,7513.0,2021-09-05 00:00:00,54.0,52.0,10521.5,547.0,82.2,19.0,22.6,17.0,11.9,3.1,541.5,85.1,60.2
max,9926.0,2021-11-14 00:00:00,68.0,74.0,29545.1,1042.0,110.2,24.0,31.3,24.7,18.4,60.3,892.0,99.3,60.2
std,2496.5,NaN,8.7,8.7,4822.7,115.7,8.2,2.2,7.1,6.3,6.3,6.9,266.3,11.2,5.8


---
## Section 1 — Per-day Feature Engineering

Derive physiological summary features from hourly data, then aggregate to participant level.

| Feature | Description |
|---------|------------|
| `resting_hr` | Mean HR during low-activity hours (steps < 50) |
| `day_hr` / `night_hr` | Mean HR during daytime (7–21h) and nighttime (22–6h) |
| `hr_day_night_delta` | `day_hr - night_hr` — circadian HR drop |
| `hr_var` | Intraday HR standard deviation |
| `hr_per_100steps` | Mean HR / mean steps × 100 during active hours — cardiac efficiency proxy |
| `sleep_fragmentation` | Number of sleep↔wake transitions (binary `sleep > 0`, then diff) |
| `sleep_onset_hour` | First hour with sleep > 0 (evening/night window 20–6h) |
| `sleep_end_hour` | Last hour with sleep > 0 in same window |


In [55]:
def _day_features(g):
    resting   = g.loc[g['steps'] < 50, 'heartrate']
    active    = g.loc[g['steps'] > 0]
    day_mask  = g['hour'].between(7, 21)
    night_mask = (g['hour'] >= 22) | (g['hour'] <= 6)
    night_sleep_mask = (g['hour'] >= 20) | (g['hour'] <= 6)
    sleep_night = g.loc[night_sleep_mask].sort_values('hour')
    sleep_hours = g.loc[night_sleep_mask & (g['sleep'] > 0), 'hour']

    day_hr   = g.loc[day_mask,   'heartrate'].mean()
    night_hr = g.loc[night_mask, 'heartrate'].mean()

    fragmentation = int((g['sleep'].gt(0).astype(int).diff().abs() > 0).sum())

    hr_per_100 = np.nan
    if len(active) >= 2:
        mean_hr_act  = active['heartrate'].mean()
        mean_step_act = active['steps'].mean()
        if mean_step_act > 0:
            hr_per_100 = mean_hr_act / mean_step_act * 100

    return pd.Series({
        'resting_hr':           resting.mean(),
        'day_hr':               day_hr,
        'night_hr':             night_hr,
        'hr_day_night_delta':   day_hr - night_hr if pd.notna(day_hr) and pd.notna(night_hr) else np.nan,
        'hr_var':               g['heartrate'].std(),
        'hr_per_100steps':      hr_per_100,
        'sleep_fragmentation':  fragmentation,
        'sleep_onset_hour':     sleep_hours.min() if len(sleep_hours) > 0 else np.nan,
        'sleep_end_hour':       sleep_hours.max() if len(sleep_hours) > 0 else np.nan,
    })

print('Computing per-day features (may take ~30s)...')
day_features = (
    df.groupby(['id', 'date'], group_keys=False)
    .apply(_day_features)
    .reset_index()
)
daily = daily.merge(day_features, on=['id', 'date'], how='left')
print(f'day_features computed for {len(day_features):,} patient-days')
print(f'New daily columns: {list(day_features.columns[2:])}')
daily[['resting_hr','day_hr','night_hr','hr_day_night_delta','hr_var',
       'hr_per_100steps','sleep_fragmentation']].describe().round(2)


Computing per-day features (may take ~30s)...
day_features computed for 1,916 patient-days
New daily columns: ['resting_hr', 'day_hr', 'night_hr', 'hr_day_night_delta', 'hr_var', 'hr_per_100steps', 'sleep_fragmentation', 'sleep_onset_hour', 'sleep_end_hour']


,resting_hr,day_hr,night_hr,hr_day_night_delta,hr_var,hr_per_100steps,sleep_fragmentation
count,1916.00,1916.00,1916.00,1916.00,1916.00,1916.00,1916.00
mean,69.48,83.89,69.84,14.05,10.40,29.73,2.13
std,8.03,8.98,8.62,6.72,2.89,22.29,0.83
min,52.11,63.34,52.48,-9.82,3.12,5.40,0.00
25%,63.70,77.85,63.90,9.74,8.35,14.28,2.00
50%,68.08,82.68,68.30,14.01,10.12,22.26,2.00
75%,73.89,88.26,74.43,18.59,12.33,37.50,2.00
max,100.97,116.87,103.84,33.16,24.81,165.30,7.00


In [56]:
patient_df = (
    daily.groupby(['id', 'disease_type', 'sex', 'age'])
    .agg(
        n_complete_days   = ('n_complete_days',    'first'),
        mean_steps        = ('daily_steps',         'mean'),
        median_steps      = ('daily_steps',         'median'),
        steps_cv          = ('daily_steps',         lambda x: x.std() / x.mean() if x.mean() > 0 else np.nan),
        mean_active_hours = ('active_hours',        'mean'),
        mean_resting_hr   = ('resting_hr',          'mean'),
        mean_hr_var       = ('hr_var',              'mean'),
        mean_hr_per_100steps = ('hr_per_100steps',  'mean'),
        mean_day_night_delta = ('hr_day_night_delta','mean'),
        mean_sleep        = ('sleep_last_night',    'mean'),
        mean_frag         = ('sleep_fragmentation', 'mean'),
    )
    .reset_index()
)
print(f'patient_df: {len(patient_df)} participants x {patient_df.shape[1]} features')
print('\nMean feature values by disease stage:')
print(
    patient_df.groupby('disease_type')[[
        'mean_steps','mean_resting_hr','mean_hr_var',
        'mean_sleep','mean_frag','mean_active_hours'
    ]].mean().round(1).to_string()
)
patient_df.head()


patient_df: 42 participants x 15 features

Mean feature values by disease stage:
                          mean_steps  mean_resting_hr  mean_hr_var  mean_sleep  mean_frag  mean_active_hours
disease_type                                                                                                
Early Disease Stage           9462.6             68.8         11.1       455.7        2.0               18.5
Fast Disease Progression      7889.0             69.9         10.7       508.7        2.4               16.8
Late Disease Stage            5967.6             69.6          9.7       492.2        2.2               17.8


,id,disease_type,sex,age,n_complete_days,mean_steps,median_steps,steps_cv,mean_active_hours,mean_resting_hr,mean_hr_var,mean_hr_per_100steps,mean_day_night_delta,mean_sleep,mean_frag
0,1120,Late Disease Stage,Female,44,52,6932.288247,7062.923184,0.248172,17.750000,79.813222,12.784330,27.307855,20.762329,396.923077,1.961538
1,1556,Early Disease Stage,Male,50,54,16834.839772,16830.220446,0.221755,20.796296,90.817099,9.370387,13.422415,12.786878,389.370370,2.074074
2,1971,Late Disease Stage,Female,55,55,2533.339521,2505.694564,0.285589,17.745455,68.300987,7.549912,57.136069,11.202452,486.709091,1.981818
3,2130,Late Disease Stage,Male,68,52,2322.516372,2308.720410,0.350309,17.019231,65.638694,9.906272,60.645287,13.994999,497.923077,2.384615
4,2589,Late Disease Stage,Male,60,53,7458.357450,6657.236367,0.633311,19.283019,69.025358,8.129092,30.054814,10.700762,503.792453,2.415094


---
## Section 2 — Individual Exploration Tools

Two interactive functions for drilling into a single patient's record:
- **`plot_patient_overview(patient_id)`** — full study timeline (steps / HR / sleep)
- **`plot_hourly_profile(patient_id)`** — mean hourly pattern across all complete days


In [57]:
def plot_patient_overview(patient_id, daily=daily):
    """3-row timeline: daily steps, HR (mean + resting), sleep minutes."""
    pat = daily[daily['id'] == patient_id].sort_values('date').copy()
    meta = pat.iloc[0]
    title = (f'Patient {patient_id} | {meta["disease_type"]} | '
             f'{meta["sex"]} | Age {int(meta["age"])} | '
             f'{int(meta["n_complete_days"])} complete days')

    has_fatigue = 'fatigue_day' in pat.columns
    step_colors = (
        pat['fatigue_day'].map({True: '#e74c3c', False: '#3498db'})
        if has_fatigue else ['#3498db'] * len(pat)
    )

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.4, 0.35, 0.25],
        vertical_spacing=0.04,
        subplot_titles=['Daily Steps', 'Heart Rate (BPM)', 'Sleep (min/day)'],
    )

    # Row 1: steps bar
    fig.add_trace(go.Bar(
        x=pat['date'], y=pat['daily_steps'],
        marker_color=step_colors,
        name='Daily Steps',
        hovertemplate='%{x|%Y-%m-%d}<br>Steps: %{y:,.0f}<extra></extra>',
    ), row=1, col=1)

    # Row 2: mean HR line + resting HR line
    fig.add_trace(go.Scatter(
        x=pat['date'], y=pat['mean_hr'],
        mode='lines+markers', name='Mean HR',
        line=dict(color='#e67e22'),
        marker=dict(size=4),
        hovertemplate='%{x|%Y-%m-%d}<br>Mean HR: %{y:.1f}<extra></extra>',
    ), row=2, col=1)
    if 'resting_hr' in pat.columns:
        fig.add_trace(go.Scatter(
            x=pat['date'], y=pat['resting_hr'],
            mode='lines+markers', name='Resting HR',
            line=dict(color='#9b59b6', dash='dot'),
            marker=dict(size=4),
            hovertemplate='%{x|%Y-%m-%d}<br>Resting HR: %{y:.1f}<extra></extra>',
        ), row=2, col=1)

    # Row 3: sleep bar
    fig.add_trace(go.Bar(
        x=pat['date'], y=pat['sleep_last_night'],
        marker_color='#1abc9c', name='Sleep',
        hovertemplate='%{x|%Y-%m-%d}<br>Sleep: %{y} min<extra></extra>',
    ), row=3, col=1)

    fig.update_layout(
        title=title, height=600, width=900,
        showlegend=True,
        legend=dict(orientation='h', y=-0.05),
        hovermode='x unified',
    )
    fig.update_yaxes(title_text='Steps',  row=1, col=1)
    fig.update_yaxes(title_text='BPM',    row=2, col=1)
    fig.update_yaxes(title_text='min',    row=3, col=1)
    fig.show()


In [58]:
def plot_hourly_profile(patient_id, df=df):
    """Mean hourly pattern across all complete days for one patient."""
    pat = df[df['id'] == patient_id].copy()
    meta = pat.iloc[0]
    hourly = pat.groupby('hour').agg(
        steps_mean    = ('steps',     'mean'),
        hr_mean       = ('heartrate', 'mean'),
        sleep_mean    = ('sleep',     'mean'),
        steps_sem     = ('steps',     lambda x: x.std() / np.sqrt(len(x))),
        hr_sem        = ('heartrate', lambda x: x.std() / np.sqrt(x.notna().sum())),
    ).reset_index()

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=['Mean Hourly Steps', 'Mean Hourly HR (BPM)', 'Mean Hourly Sleep (min)'],
        vertical_spacing=0.08,
    )

    # Steps with shaded SEM
    fig.add_trace(go.Scatter(
        x=hourly['hour'], y=hourly['steps_mean'],
        mode='lines', name='Steps', line=dict(color='#3498db'),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=pd.concat([hourly['hour'], hourly['hour'][::-1]]),
        y=pd.concat([
            hourly['steps_mean'] + hourly['steps_sem'],
            (hourly['steps_mean'] - hourly['steps_sem'])[::-1],
        ]),
        fill='toself', fillcolor='rgba(52,152,219,0.2)', line=dict(color='rgba(0,0,0,0)'),
        showlegend=False, name='Steps SEM',
    ), row=1, col=1)

    # HR with shaded SEM
    fig.add_trace(go.Scatter(
        x=hourly['hour'], y=hourly['hr_mean'],
        mode='lines', name='HR', line=dict(color='#e67e22'),
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=pd.concat([hourly['hour'], hourly['hour'][::-1]]),
        y=pd.concat([
            hourly['hr_mean'] + hourly['hr_sem'],
            (hourly['hr_mean'] - hourly['hr_sem'])[::-1],
        ]),
        fill='toself', fillcolor='rgba(230,126,34,0.2)', line=dict(color='rgba(0,0,0,0)'),
        showlegend=False,
    ), row=2, col=1)

    # Sleep bar
    fig.add_trace(go.Bar(
        x=hourly['hour'], y=hourly['sleep_mean'],
        marker_color='#1abc9c', name='Sleep',
    ), row=3, col=1)

    # Night shading on all rows
    for row in [1, 2, 3]:
        for x0, x1 in [(0, 6.5), (22, 23)]:
            fig.add_vrect(x0=x0, x1=x1, fillcolor='navy', opacity=0.06,
                          line_width=0, row=row, col=1)

    fig.update_xaxes(tickmode='array', tickvals=list(range(0,24,3)),
                     ticktext=[f'{h:02d}:00' for h in range(0,24,3)])
    fig.update_yaxes(title_text='Steps', row=1, col=1)
    fig.update_yaxes(title_text='BPM',   row=2, col=1)
    fig.update_yaxes(title_text='min',   row=3, col=1)
    fig.update_layout(
        title=f'Patient {patient_id} — Mean Hourly Profile | '
              f'{meta["disease_type"]} | {meta["sex"]} | Age {int(meta["age"])}',
        height=620, width=820, showlegend=True,
        legend=dict(orientation='h', y=-0.06),
    )
    fig.show()


In [59]:
# Demo with patient 3389 — change patient_id to explore others
DEMO_ID = 3389
plot_patient_overview(DEMO_ID)
plot_hourly_profile(DEMO_ID)


> **Tip:** The trellis week × day-of-week plot from `eda2_overall` works here too:
> copy-paste the `plot_patient_timeseries(patient_id, 'steps')` function from that notebook
> if you want to see the full hourly grid for a specific patient.


---
## Section 4 — Phase Analysis (First vs Second Half of Study)

Intervention / follow-up labels are **not available** in the dataset. As a proxy we split each patient's complete days at the chronological midpoint:
- **Phase 1** = first half of the patient's recording period
- **Phase 2** = second half

This lets us ask: *Did step counts, heart rate, or sleep improve or worsen across the study?*


In [67]:
# Complete-day coverage chart: y = patient, x = relative calendar day
# (day 1 = each patient's first complete day; white gaps = calendar days with missing data)
# Colour = disease stage; hover shows total complete days for that patient.

_first = daily.groupby('id')['date'].min().rename('first_date').reset_index()
daily_rd = daily.merge(_first, on='id')
daily_rd['relative_day'] = (daily_rd['date'] - daily_rd['first_date']).dt.days + 1

_stage_abbr = {
    'Early Disease Stage':      'Early',
    'Fast Disease Progression': 'Fast',
    'Late Disease Stage':       'Late',
}
_id_order = (
    patient_df
    .assign(_r=patient_df['disease_type'].map({s: i for i, s in enumerate(DISEASE_ORDER)}))
    .sort_values(['_r', 'n_complete_days'], ascending=[True, False])['id']
    .tolist()
)
_y_pos  = {pid: i for i, pid in enumerate(_id_order)}
_y_labels = [
    f'{pid} ({_stage_abbr.get(patient_df.loc[patient_df["id"]==pid, "disease_type"].values[0], "?")})'  
    for pid in _id_order
]

# Long-form: one row per (patient, relative_day)
_long = (
    daily_rd[['id', 'relative_day', 'disease_type']]
    .merge(patient_df[['id', 'n_complete_days']], on='id')
    .assign(y_pos=lambda d: d['id'].map(_y_pos))
)

fig = go.Figure()
for stage in DISEASE_ORDER:
    sub = _long[_long['disease_type'] == stage]
    fig.add_trace(go.Scatter(
        x=sub['relative_day'],
        y=sub['y_pos'],
        mode='markers',
        marker=dict(symbol='square', size=9, color=DISEASE_PALETTE[stage]),
        name=stage,
        customdata=sub[['n_complete_days', 'id']].values,
        hovertemplate=(
            'Patient %{customdata[1]}<br>'
            'Relative day: %{x}<br>'
            'Total complete days: %{customdata[0]:.0f}'
            '<extra></extra>'
        ),
    ))

# Horizontal dividers between disease stage groups
_stage_sizes = [
    sum(1 for pid in _id_order
        if patient_df.loc[patient_df['id'] == pid, 'disease_type'].values[0] == s)
    for s in DISEASE_ORDER
]
_cum = 0
for _s, _cnt in zip(DISEASE_ORDER, _stage_sizes):
    fig.add_annotation(
        x=daily_rd['relative_day'].max() + 1,
        y=_cum + _cnt / 2 - 0.5,
        text=_stage_abbr[_s], showarrow=False,
        font=dict(size=10, color=DISEASE_PALETTE[_s]),
        xanchor='left',
    )
    _cum += _cnt
    if _cum < len(_id_order):
        fig.add_hline(y=_cum - 0.5, line_width=1.2, line_color='grey', opacity=0.4)

fig.update_layout(
    title="Patient Complete-day Coverage (relative calendar day — gaps = days with missing data)",
    xaxis_title="Relative Study Day (day 1 = each patient's first complete day)",
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(_id_order))),
        ticktext=_y_labels,
        tickfont=dict(size=9),
        title='Patient ID',
    ),
    height=max(380, 16 * len(_id_order)),
    width=1050,
    legend=dict(orientation='h', y=1.03, x=0),
    margin=dict(r=90),
    plot_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridcolor='#eeeeee')
fig.update_yaxes(showgrid=False)
fig.show()

print(f'Max relative day span: {daily_rd["relative_day"].max()}')
print(f'Mean complete days per patient: {daily_rd.groupby("id")["relative_day"].nunique().mean():.1f}')


Max relative day span: 96
Mean complete days per patient: 45.6


In [68]:
daily = daily.sort_values(['id', 'date']).copy()
daily['day_rank'] = daily.groupby('id').cumcount() + 1
daily['n_days']   = daily.groupby('id')['date'].transform('nunique')

# Leave 7 days (1 week) in the middle unassigned as a washout buffer.
# Buffer is centred on the midpoint: days [mid-3 .. mid+3] get NaN.
# Phase 1 = day_rank <= mid-4,  Phase 2 = day_rank >= mid+4.
_mid = daily['n_days'] // 2
daily['phase'] = np.select(
    [
        daily['day_rank'] <= _mid - 3,
        daily['day_rank'] >= _mid + 4,
    ],
    ['Phase 1', 'Phase 2'],
    default=None,          # buffer days
)

n_buf = daily['phase'].isna().sum()
print(f'Buffer (washout) days: {n_buf:,} rows '
      f'({n_buf / len(daily):.1%} of all patient-days)')
print()
print('Phase assignment summary (buffer excluded):')
print(daily.groupby(['disease_type', 'phase'])['id'].count().unstack().reindex(DISEASE_ORDER).to_string())
print()
print('Mean steps by phase:')
print(
    daily.groupby(['disease_type', 'phase'])['daily_steps']
    .mean().round(0).unstack().reindex(DISEASE_ORDER).to_string()
)


Buffer (washout) days: 252 rows (13.2% of all patient-days)

Phase assignment summary (buffer excluded):
phase                     Phase 1  Phase 2
disease_type                              
Early Disease Stage           298      303
Fast Disease Progression      132      137
Late Disease Stage            392      402

Mean steps by phase:
phase                     Phase 1  Phase 2
disease_type                              
Early Disease Stage       10564.0   9317.0
Fast Disease Progression   8425.0   7604.0
Late Disease Stage         6797.0   4905.0


In [69]:
# Same relative-day chart as above, now coloured by phase assignment.
# Blue = Phase 1, Orange = Phase 2, Grey = buffer (washout week), White gap = missing data.

_first2 = daily.groupby('id')['date'].min().rename('first_date').reset_index()
daily_rd2 = daily.merge(_first2, on='id')
daily_rd2['relative_day'] = (daily_rd2['date'] - daily_rd2['first_date']).dt.days + 1
daily_rd2['phase_label'] = daily_rd2['phase'].fillna('Buffer')

_id_order2 = (
    patient_df
    .assign(_r=patient_df['disease_type'].map({s: i for i, s in enumerate(DISEASE_ORDER)}))
    .sort_values(['_r', 'n_complete_days'], ascending=[True, False])['id']
    .tolist()
)
_y_pos2 = {pid: i for i, pid in enumerate(_id_order2)}
_stage_abbr2 = {
    'Early Disease Stage': 'Early',
    'Fast Disease Progression': 'Fast',
    'Late Disease Stage': 'Late',
}
_y_labels2 = [
    f'{pid} ({_stage_abbr2.get(patient_df.loc[patient_df["id"]==pid, "disease_type"].values[0], "?")})'  
    for pid in _id_order2
]

PHASE_PALETTE = {
    'Phase 1': '#3498db',
    'Phase 2': '#e67e22',
    'Buffer':  '#bbbbbb',
}

daily_rd2['y_pos'] = daily_rd2['id'].map(_y_pos2)
daily_rd2 = daily_rd2.merge(patient_df[['id', 'n_complete_days']], on='id', suffixes=('', '_y'))

fig = go.Figure()
for label in ['Phase 1', 'Buffer', 'Phase 2']:
    sub = daily_rd2[daily_rd2['phase_label'] == label]
    fig.add_trace(go.Scatter(
        x=sub['relative_day'],
        y=sub['y_pos'],
        mode='markers',
        marker=dict(symbol='square', size=9, color=PHASE_PALETTE[label]),
        name=label,
        customdata=sub[['n_complete_days', 'id']].values,
        hovertemplate=(
            'Patient %{customdata[1]}<br>'
            'Relative day: %{x}<br>'
            f'Phase: {label}<br>'
            'Total complete days: %{customdata[0]:.0f}'
            '<extra></extra>'
        ),
    ))

# Disease stage dividers and labels
_stage_sizes2 = [
    sum(1 for pid in _id_order2
        if patient_df.loc[patient_df['id'] == pid, 'disease_type'].values[0] == s)
    for s in DISEASE_ORDER
]
_cum2 = 0
for _s, _cnt in zip(DISEASE_ORDER, _stage_sizes2):
    fig.add_annotation(
        x=daily_rd2['relative_day'].max() + 1,
        y=_cum2 + _cnt / 2 - 0.5,
        text=_stage_abbr2[_s], showarrow=False,
        font=dict(size=10, color=DISEASE_PALETTE[_s]),
        xanchor='left',
    )
    _cum2 += _cnt
    if _cum2 < len(_id_order2):
        fig.add_hline(y=_cum2 - 0.5, line_width=1.2, line_color='grey', opacity=0.4)

fig.update_layout(
    title="Patient Coverage with Phase Assignment (Blue=Phase 1, Orange=Phase 2, Grey=Buffer)",
    xaxis_title="Relative Study Day (day 1 = each patient's first complete day)",
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(_id_order2))),
        ticktext=_y_labels2,
        tickfont=dict(size=9),
        title='Patient ID',
    ),
    legend=dict(orientation='h', y=1.03, x=0),
    height=max(380, 16 * len(_id_order2)),
    width=1050,
    margin=dict(r=90),
    plot_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridcolor='#eeeeee')
fig.update_yaxes(showgrid=False)
fig.show()


In [70]:
# Per-participant mean steps in each phase
pp_phase = (
    daily.groupby(['id', 'disease_type', 'sex', 'age', 'phase'])
    .agg(mean_steps=('daily_steps','mean'), mean_resting_hr=('resting_hr','mean'),
         mean_sleep=('sleep_last_night','mean'), mean_active=('active_hours','mean'))
    .reset_index()
)

fig = px.box(
    pp_phase, x='disease_type', y='mean_steps',
    color='phase', color_discrete_map={'Phase 1': '#3498db', 'Phase 2': '#e74c3c'},
    category_orders={'disease_type': DISEASE_ORDER, 'phase': ['Phase 1', 'Phase 2']},
    points='all',
    hover_name='id',
    hover_data={'sex': True, 'age': True, 'disease_type': False},
    labels={'disease_type': 'Disease Stage', 'mean_steps': 'Mean Daily Steps', 'phase': 'Study Phase'},
    title='Finding D: Mean Daily Steps — Phase 1 vs Phase 2 by Disease Stage',
    width=860, height=500,
)
fig.update_traces(boxmean=True)
fig.show()

# Paired t-test per disease group
print('Paired t-tests (Phase 1 vs Phase 2 per participant):')
for stage in DISEASE_ORDER:
    sub   = pp_phase[pp_phase['disease_type'] == stage]
    pivot = sub.pivot_table(index='id', columns='phase', values='mean_steps').dropna()
    if len(pivot) >= 3:
        t, p = stats.ttest_rel(pivot['Phase 1'], pivot['Phase 2'])
        delta = pivot['Phase 2'].mean() - pivot['Phase 1'].mean()
        print(f'  {stage[:20]:20s}: t={t:.2f}, p={p:.3f}, delta={delta:+.0f} steps  (n={len(pivot)})')
    else:
        print(f'  {stage[:20]:20s}: too few participants ({len(pivot)})')


Paired t-tests (Phase 1 vs Phase 2 per participant):
  Early Disease Stage : t=1.41, p=0.179, delta=-1085 steps  (n=16)
  Fast Disease Progres: t=1.00, p=0.355, delta=-1157 steps  (n=7)
  Late Disease Stage  : t=3.52, p=0.002, delta=-1833 steps  (n=19)


In [71]:
fig = px.box(
    pp_phase.dropna(subset=['mean_resting_hr']),
    x='disease_type', y='mean_resting_hr',
    color='phase', color_discrete_map={'Phase 1': '#3498db', 'Phase 2': '#e74c3c'},
    category_orders={'disease_type': DISEASE_ORDER, 'phase': ['Phase 1', 'Phase 2']},
    points='all', hover_name='id',
    hover_data={'sex': True, 'age': True, 'disease_type': False},
    labels={'disease_type': 'Disease Stage', 'mean_resting_hr': 'Mean Resting HR (BPM)', 'phase': 'Study Phase'},
    title='Finding E: Mean Resting HR — Phase 1 vs Phase 2 by Disease Stage',
    width=860, height=500,
)
fig.update_traces(boxmean=True)
fig.show()

print('Paired t-tests (resting HR, Phase 1 vs Phase 2):')
for stage in DISEASE_ORDER:
    sub   = pp_phase[pp_phase['disease_type'] == stage]
    pivot = sub.pivot_table(index='id', columns='phase', values='mean_resting_hr').dropna()
    if len(pivot) >= 3:
        t, p = stats.ttest_rel(pivot['Phase 1'], pivot['Phase 2'])
        delta = pivot['Phase 2'].mean() - pivot['Phase 1'].mean()
        print(f'  {stage[:20]:20s}: t={t:.2f}, p={p:.3f}, delta={delta:+.1f} BPM  (n={len(pivot)})')
    else:
        print(f'  {stage[:20]:20s}: too few participants ({len(pivot)})')


Paired t-tests (resting HR, Phase 1 vs Phase 2):
  Early Disease Stage : t=-1.42, p=0.176, delta=+1.6 BPM  (n=16)
  Fast Disease Progres: t=-1.69, p=0.141, delta=+2.5 BPM  (n=7)
  Late Disease Stage  : t=-2.47, p=0.024, delta=+1.6 BPM  (n=19)


In [72]:
fig = px.box(
    pp_phase, x='disease_type', y='mean_sleep',
    color='phase', color_discrete_map={'Phase 1': '#3498db', 'Phase 2': '#e74c3c'},
    category_orders={'disease_type': DISEASE_ORDER, 'phase': ['Phase 1', 'Phase 2']},
    points='all', hover_name='id',
    hover_data={'sex': True, 'age': True, 'disease_type': False},
    labels={'disease_type': 'Disease Stage', 'mean_sleep': 'Mean Sleep Last Night (min)', 'phase': 'Study Phase'},
    title='Finding F: Mean Sleep Last Night — Phase 1 vs Phase 2 by Disease Stage',
    width=860, height=500,
)
fig.update_traces(boxmean=True)
fig.show()


In [73]:
summary = (
    pp_phase.groupby(['disease_type', 'phase'])[['mean_steps','mean_resting_hr','mean_sleep']]
    .agg(['mean','std'])
    .round(1)
    .reindex(DISEASE_ORDER, level=0)
)
summary.columns = [' '.join(c) for c in summary.columns]
print('Phase summary table (mean ± std):')
summary


Phase summary table (mean ± std):


mean_steps mean  mean_steps std  \
disease_type             phase                                      
Early Disease Stage      Phase 1          10022.0          4396.5   
                         Phase 2           8937.3          4440.6   
Fast Disease Progression Phase 1           8493.2          3127.4   
                         Phase 2           7336.5          3917.8   
Late Disease Stage       Phase 1           6922.7          2723.0   
                         Phase 2           5089.8          2899.6   

                                  mean_resting_hr mean  mean_resting_hr std  \
disease_type             phase                                                
Early Disease Stage      Phase 1                  68.2                 10.1   
                         Phase 2                  69.8                  8.5   
Fast Disease Progression Phase 1                  68.4                  9.5   
                         Phase 2                  70.9                  9.4   
Late Disease Stage       Phase 1                  68.8                  5.5   
                         Phase 2                  70.4                  5.3   

                                  mean_sleep mean  mean_sleep std  
disease_type             phase                                     
Early Disease Stage      Phase 1            463.3            89.7  
                         Phase 2            446.0            92.7  
Fast Disease Progression Phase 1            488.6            51.5  
                         Phase 2            525.2           115.0  
Late Disease Stage       Phase 1            492.3            62.9  
                         Phase 2            493.7            55.9

---
## Section 5 — Cross-patient Feature Comparison

Two complementary views over the participant-level feature matrix:
- **Radar chart** — per-disease-stage mean profile across 6 normalized features
- **Scatter matrix** — pairwise correlations between features, colored by disease stage


In [74]:
# Radar chart: 6 features, min-max normalized across all patients
RADAR_FEATURES = ['mean_steps', 'mean_active_hours', 'mean_resting_hr',
                   'mean_hr_var', 'mean_sleep', 'mean_frag']
RADAR_LABELS   = ['Daily Steps', 'Active Hours', 'Resting HR',
                   'HR Variability', 'Sleep (min)', 'Sleep Fragmentation']

norm = patient_df[RADAR_FEATURES].copy()
for col in RADAR_FEATURES:
    lo, hi = norm[col].min(), norm[col].max()
    norm[col] = (norm[col] - lo) / (hi - lo) if hi > lo else 0.5
norm['disease_type'] = patient_df['disease_type']

stage_means = norm.groupby('disease_type')[RADAR_FEATURES].mean().reindex(DISEASE_ORDER)

fig = go.Figure()
for stage in DISEASE_ORDER:
    vals = stage_means.loc[stage].tolist()
    vals += [vals[0]]  # close the polygon
    lbls = RADAR_LABELS + [RADAR_LABELS[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=lbls,
        fill='toself',
        name=stage,
        line_color=DISEASE_PALETTE[stage],
        fillcolor=DISEASE_RGBA[stage],
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Finding G: Patient Feature Radar — Disease Stage Profiles (min-max normalized)',
    width=650, height=550,
    showlegend=True,
)
fig.show()


In [26]:
SCATTER_FEATURES = ['mean_steps', 'mean_active_hours', 'mean_resting_hr',
                     'mean_hr_var', 'mean_sleep', 'mean_frag', 'steps_cv']
SCATTER_LABELS   = ['Steps', 'Active Hrs', 'Resting HR',
                     'HR Var', 'Sleep', 'Fragmentation', 'Steps CV']

fig = px.scatter_matrix(
    patient_df.dropna(subset=SCATTER_FEATURES),
    dimensions=SCATTER_FEATURES,
    labels={f: l for f, l in zip(SCATTER_FEATURES, SCATTER_LABELS)},
    color='disease_type',
    color_discrete_map=DISEASE_PALETTE,
    category_orders={'disease_type': DISEASE_ORDER},
    symbol='sex',
    hover_name='id',
    hover_data={'age': True},
    title='Finding H: Participant Feature Scatter Matrix (each point = one patient)',
    width=900, height=900,
)
fig.update_traces(diagonal_visible=False, marker_size=7, opacity=0.8)
fig.show()
print('Spearman correlation matrix:')
print(patient_df[SCATTER_FEATURES].corr(method='spearman').round(2).to_string())


Spearman correlation matrix:
                   mean_steps  mean_active_hours  mean_resting_hr  mean_hr_var  mean_sleep  mean_frag  steps_cv
mean_steps               1.00               0.09            -0.15         0.45       -0.02      -0.36     -0.35
mean_active_hours        0.09               1.00             0.14         0.12       -0.58      -0.36     -0.09
mean_resting_hr         -0.15               0.14             1.00        -0.19       -0.25       0.25      0.18
mean_hr_var              0.45               0.12            -0.19         1.00       -0.21      -0.43     -0.26
mean_sleep              -0.02              -0.58            -0.25        -0.21        1.00       0.30      0.14
mean_frag               -0.36              -0.36             0.25        -0.43        0.30       1.00      0.43
steps_cv                -0.35              -0.09             0.18        -0.26        0.14       0.43      1.00


---
## Section 6 — Advanced Individual Patterns

| Finding | Question | Method |
|---------|----------|-------|
| **I** | Push-crash: does a very active day predict a low-activity rebound? | Lag-1 step autocorrelation per patient |
| **J** | Uhthoff's: are MS patients more sensitive to heat? | OLS slope of steps ~ temp_max per patient |
| **K** | Sleep–activity coupling: does poor sleep predict next-day fatigue? | Lagged Spearman r (sleep_t → steps_t+1) per patient |
| **L** | Recovery speed: how long after a fatigue day to return to baseline? | Median recovery lag by disease stage |


### Finding I — Push-Crash Pattern

MS-related fatigue often manifests as a **boom-bust cycle**: an unusually active day is followed by a low-activity 'crash' day.
We test this with two complementary analyses:
- **Lag-1 autocorrelation** of daily steps per patient (negative = systematic rebound)
- **Conditional next-day steps**: mean steps on day *t+1* after a top-quartile day vs all other days


In [75]:
# Lag-1 autocorrelation of daily steps per patient
def _lag1_autocorr(series):
    s = series.dropna()
    if len(s) < 5:
        return np.nan
    return s.autocorr(lag=1)

daily_sorted = daily.sort_values(['id', 'date']).copy()
daily_sorted['steps_lag1'] = daily_sorted.groupby('id')['daily_steps'].shift(1)

lag1_corr = (
    daily_sorted.groupby(['id', 'disease_type'])['daily_steps']
    .apply(_lag1_autocorr)
    .reset_index()
    .rename(columns={'daily_steps': 'lag1_autocorr'})
)

fig = px.box(
    lag1_corr, x='disease_type', y='lag1_autocorr',
    color='disease_type', color_discrete_map=DISEASE_PALETTE,
    category_orders={'disease_type': DISEASE_ORDER},
    points='all',
    hover_name='id',
    labels={'disease_type': 'Disease Stage', 'lag1_autocorr': 'Lag-1 Autocorrelation (steps)'},
    title='Finding I-a: Lag-1 Step Autocorrelation by Disease Stage'
          ' (negative = push-crash pattern)',
    width=760, height=460,
)
fig.add_hline(y=0, line_dash='dash', line_color='grey', opacity=0.6)
fig.update_traces(marker_size=8)
fig.show()

print('Mean lag-1 autocorrelation by disease stage:')
print(lag1_corr.groupby('disease_type')['lag1_autocorr'].describe().round(3).to_string())
print()

# One-sample t-test: is mean autocorr < 0?
for stage in DISEASE_ORDER:
    vals = lag1_corr.loc[lag1_corr['disease_type']==stage, 'lag1_autocorr'].dropna()
    t, p = stats.ttest_1samp(vals, 0)
    print(f'{stage[:20]:22s}: mean={vals.mean():.3f}, t={t:.2f}, p={p:.3f} (n={len(vals)})')


Mean lag-1 autocorrelation by disease stage:
                          count   mean    std    min    25%    50%    75%    max
disease_type                                                                    
Early Disease Stage        16.0  0.267  0.280 -0.147  0.014  0.260  0.475  0.683
Fast Disease Progression    7.0  0.190  0.201 -0.067  0.069  0.181  0.275  0.528
Late Disease Stage         19.0  0.357  0.270 -0.068  0.154  0.387  0.593  0.826

Early Disease Stage   : mean=0.267, t=3.81, p=0.002 (n=16)
Fast Disease Progres  : mean=0.190, t=2.51, p=0.046 (n=7)
Late Disease Stage    : mean=0.357, t=5.77, p=0.000 (n=19)


In [76]:
# Conditional rebound: mean next-day steps after a top-quartile day vs all other days
q75 = daily_sorted.groupby('id')['daily_steps'].transform(lambda x: x.quantile(0.75))
daily_sorted['high_day']      = daily_sorted['daily_steps'] >= q75
daily_sorted['next_day_steps'] = daily_sorted.groupby('id')['daily_steps'].shift(-1)

rebound = daily_sorted.dropna(subset=['next_day_steps']).copy()
rebound['prev_day_type'] = rebound['high_day'].map({True: 'After high-activity day (top 25%)',
                                                      False: 'After normal/low day'})

fig = px.box(
    rebound, x='disease_type', y='next_day_steps',
    color='prev_day_type',
    color_discrete_map={
        'After high-activity day (top 25%)': '#e74c3c',
        'After normal/low day':              '#3498db',
    },
    category_orders={
        'disease_type': DISEASE_ORDER,
        'prev_day_type': ['After high-activity day (top 25%)', 'After normal/low day'],
    },
    points=False,
    labels={
        'disease_type':   'Disease Stage',
        'next_day_steps': 'Next-day Steps',
        'prev_day_type':  'Previous Day',
    },
    title='Finding I-b: Next-day Steps After a High-Activity Day vs Normal Day',
    width=860, height=460,
)
fig.show()

print('Mean next-day steps (high-activity day vs other):')
print(
    rebound.groupby(['disease_type','prev_day_type'])['next_day_steps']
    .mean().round(0).unstack().reindex(DISEASE_ORDER).to_string()
)


Mean next-day steps (high-activity day vs other):
prev_day_type             After high-activity day (top 25%)  After normal/low day
disease_type                                                                     
Early Disease Stage                                 10850.0                9458.0
Fast Disease Progression                             8753.0                7695.0
Late Disease Stage                                   7103.0                5379.0


### Finding J — Heat Sensitivity (Uhthoff's Phenomenon)

MS patients often experience transient symptom worsening in heat (*Uhthoff's phenomenon*).
We estimate each patient's **temperature sensitivity** as the OLS regression slope of `daily_steps ~ temp_max`.
A steeper negative slope indicates greater activity suppression on hot days.


In [77]:
from scipy.stats import linregress

def _temp_slope(sub):
    s = sub.dropna(subset=['daily_steps', 'temp_max'])
    if len(s) < 5:
        return pd.Series({'slope': np.nan, 'r': np.nan, 'p': np.nan})
    res = linregress(s['temp_max'], s['daily_steps'])
    return pd.Series({'slope': res.slope, 'r': res.rvalue, 'p': res.pvalue})

heat_sens = (
    daily.groupby(['id', 'disease_type', 'sex', 'age'])
    .apply(_temp_slope)
    .reset_index()
)

fig = px.box(
    heat_sens.dropna(subset=['slope']),
    x='disease_type', y='slope',
    color='disease_type', color_discrete_map=DISEASE_PALETTE,
    category_orders={'disease_type': DISEASE_ORDER},
    points='all', hover_name='id',
    hover_data={'sex': True, 'age': True, 'r': ':.3f', 'p': ':.3f', 'disease_type': False},
    labels={'disease_type': 'Disease Stage', 'slope': 'Steps per °C (OLS slope)'},
    title='Finding J: Temperature Sensitivity — Steps per 1°C Increase in Daily Max Temp\n'
          '(negative = fewer steps on hot days)',
    width=760, height=480,
)
fig.add_hline(y=0, line_dash='dash', line_color='grey', opacity=0.6)
fig.update_traces(marker_size=8)
fig.show()

print('Mean temperature slope (steps per °C) by disease stage:')
print(heat_sens.groupby('disease_type')['slope'].describe().round(1).to_string())
print()

# Kruskal-Wallis across all three groups
groups = [heat_sens.loc[heat_sens['disease_type']==s, 'slope'].dropna() for s in DISEASE_ORDER]
h, p = stats.kruskal(*[g for g in groups if len(g) >= 3])
print(f'Kruskal-Wallis across disease stages: H={h:.2f}, p={p:.3f}')


Mean temperature slope (steps per °C) by disease stage:
                          count  mean    std    min   25%   50%    75%    max
disease_type                                                                 
Early Disease Stage        16.0  50.6  186.9 -372.4 -11.1  43.4  185.0  345.0
Fast Disease Progression    7.0  53.5  127.8  -81.7  -5.6  -1.0   97.1  274.3
Late Disease Stage         19.0  69.8  172.6 -248.0 -40.3  60.4  141.6  485.2

Kruskal-Wallis across disease stages: H=0.18, p=0.915


In [30]:
# Scatter: temp_max vs daily_steps per disease stage with OLS trend lines
fig = px.scatter(
    daily.dropna(subset=['temp_max', 'daily_steps']),
    x='temp_max', y='daily_steps',
    color='disease_type', color_discrete_map=DISEASE_PALETTE,
    category_orders={'disease_type': DISEASE_ORDER},
    opacity=0.25, trendline='ols',
    trendline_scope='trace',
    hover_data={'id': True, 'date': '|%Y-%m-%d', 'disease_type': False},
    labels={
        'temp_max': 'Daily Max Temperature (°C)',
        'daily_steps': 'Daily Steps',
        'disease_type': 'Disease Stage',
    },
    title='Finding J (scatter): Daily Steps vs Max Temperature by Disease Stage',
    width=820, height=480,
)
fig.show()
